In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil
import tarfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
from tqdm.notebook import tqdm as tqdm_bar

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [3]:
main_data_path = '/content/drive/MyDrive/Comp1/data'

In [5]:
class MyDataset(Dataset):
    def __init__(self, data_path, exec_type='train', transform=None):
        self.transform = transform

        self.samples = []
        exec_type_path = os.path.join(data_path, exec_type)

        if exec_type == 'test':
            for fname in sorted(os.listdir(exec_type_path)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(exec_type_path, fname), -1))
        else:
            for class_name in sorted(os.listdir(exec_type_path), key=lambda x: int(x)):
                class_dir = os.path.join(exec_type_path, class_name)

                if os.path.isdir(class_dir):
                    label = int(class_name)

                    for fname in os.listdir(class_dir):
                        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                            self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

In [6]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.33, saturation=0.24, hue=0.12),
    transforms.RandomRotation(22),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [21]:
dataset_train = MyDataset(main_data_path, exec_type='train', transform=train_transform)
dataset_validation   = MyDataset(main_data_path, exec_type='val',   transform=val_transform)
dataset_test  = MyDataset(main_data_path, exec_type='test',  transform=val_transform)

print(f'Train: {len(dataset_train)} Val: {len(dataset_validation)} Test: {len(dataset_test)}')

labels = [s[1] for s in dataset_train.samples]
class_counts = np.bincount(labels, minlength=100)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in labels], dtype=torch.float)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(dataset_train, batch_size=128, sampler=sampler,  num_workers=8, pin_memory=True)
val_loader   = DataLoader(dataset_validation,   batch_size=128, shuffle=False,    num_workers=8, pin_memory=True)
test_loader  = DataLoader(dataset_test,  batch_size=128, shuffle=False,    num_workers=8, pin_memory=True)

print(f'Train batches: {len(train_loader)} Val batches: {len(val_loader)}')

Train: 20724 Val: 300 Test: 2344
Train batches: 162 Val batches: 3


In [ ]:
class ModifiedResNet(nn.Module):
    def __init__(self, num_classes=100, dropout=0.4):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),   
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(1024, 512), 
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)


model = ModifiedResNet(num_classes=100).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

Total parameters: 26,185,380


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.7)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)

scaler    = torch.amp.GradScaler('cuda')

In [25]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm_bar(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for imgs, labels in tqdm_bar(loader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

In [ ]:
EPOCHS = 15
SAVE_PATH = '/content/drive/MyDrive/Comp1/best_model.pth'

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

best_val_acc = 0
history = []

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)
    scheduler.step()

    history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'Best model saved (val_acc={val_acc:.4f})')

print(f'\nTraining complete. Best val acc: {best_val_acc:.4f}')

  0%|          | 0/162 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch 01/15 | Train Loss: 4.3007  Acc: 0.4623 | Val Loss: 4.1281  Acc: 0.5967
  ✓ Best model saved (val_acc=0.5967)


  0%|          | 0/162 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

...

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch 14/15 | Train Loss: 3.8527  Acc: 0.9770 | Val Loss: 3.9208  Acc: 0.8467


  0%|          | 0/162 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch 15/15 | Train Loss: 3.8507  Acc: 0.9798 | Val Loss: 3.9213  Acc: 0.8567

Training complete. Best val acc: 0.8700


In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()

all_preds = []
all_fnames = [os.path.splitext(os.path.basename(s[0]))[0] for s in dataset_test.samples]

with torch.no_grad():
    for imgs, _ in tqdm_bar(test_loader):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda'):
            preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)

PRED_PATH = '/content/drive/MyDrive/Comp1/prediction.csv'
df = pd.DataFrame({'image_name': all_fnames, 'pred_label': all_preds})

# df = df.sort_values('image_name').reset_index(drop=True)
df.to_csv(PRED_PATH, index=False)
print(f'Predictions saved to {PRED_PATH}')
print(df.head(10))

  0%|          | 0/19 [00:00<?, ?it/s]

Predictions saved to /content/drive/MyDrive/Comp1/prediction.csv
                             image_name  pred_label
0  001a74bd-6679-4709-aa0b-d10277f057e6          33
1  002fe951-e857-4ebf-8de4-53c89b9f324e          91
2  0042dfce-6c3e-4528-a306-5ac3c1cb6bb8          39
3  008e97fa-a96b-49f1-9c23-554ffb5ba0f0          10
4  00a10414-db4c-44c8-a5dd-ca47ee4d37f4          34
5  00bbaead-4a43-4ee5-84f9-9b92589f0e67          53
6  010bd2ad-2fda-47a7-84cb-8a0dad7b6123          35
7  013d9874-2638-46a7-b32e-1b37c6435f11          11
8  0179e450-8f31-4122-be0f-da4a47a207b1          82
9  01928ab5-e908-4af2-9510-4488f620254b          84
